<h1>🔎 Biofilter — Report: <code>entity_filter</code></h1>

Does the bundle know these names, and unambiguously?

Run this **before** any other report. It tells you which of your inputs
will resolve, which are ambiguous, and which the bundle has never heard
of — the three things that quietly distort every downstream result.

### 1. Open a bundle

In [ ]:
from biofilter import Biofilter

# Leave as None to use `[database] bundle` from .biofilter.toml.
BUNDLE = None
REPORT = "entity_filter"

bf = Biofilter(bundle=BUNDLE, debug_mode=False) if BUNDLE else Biofilter(debug_mode=False)
bf

### 2. What the report offers

In [ ]:
print("columns:")
for column in bf.report.available_columns(REPORT):
    print(" ", column)

print("\nexample input:")
print(bf.report.example_input(REPORT))

In [ ]:
print(bf.report.explain(REPORT))

### 3. Run it

One row per **match**, not per input. An input matching three entities
gives three rows — that is the answer, not a problem to hide.

In [ ]:
names = ["TP53", "brca1", "NOT_A_GENE"]

result = bf.report.run(REPORT, input_data=names)
df = result.to_pandas()
df[["input_original", "input", "entity_id", "primary_name", "group_name",
    "is_primary", "observation"]]

### 4. The three things to look at

| `observation` | what it means |
| --- | --- |
| `not found` | the bundle has no entity answering to this name |
| `multiple matches` | the **name** belongs to more than one entity |
| *(empty)* | one entity, unambiguously |

`not found` rows are kept deliberately: dropping them would leave no way
to tell "the bundle does not know this" from "you did not ask".

In [ ]:
df["observation"].value_counts(dropna=False)

`multiple matches` is about the name, not your search. It means
resolving that alias requires a decision **you** have to make — which is
worth knowing before a downstream report picks one for you.

In [ ]:
ambiguous = df[df["observation"] == "multiple matches"]
ambiguous[["input_original", "input", "entity_id", "primary_name", "group_name"]]

### 5. Match modes

| mode | matches when | cost |
| --- | --- | --- |
| `exact` | the alias equals the input, case-insensitively | an equality join |
| `like` | the input occurs **inside** the alias | a scan with a substring test |
| `fuzzy` | Jaro-Winkler similarity ≥ threshold | a scan with a scored test |

In [ ]:
import time

for mode in ("exact", "like", "fuzzy"):
    started = time.perf_counter()
    out = bf.report.run(REPORT, input_data=["BRCA1"], match_mode=mode).to_pandas()
    print(f"{mode:6s} {len(out):>5,} rows in {time.perf_counter() - started:.2f}s")

`like` is one-directional on purpose: the input inside the alias, not
the reverse. Matching an alias inside an input would make every
one-character alias match every input containing that character.

In [ ]:
bf.report.run(REPORT, input_data=["BRCA1"], match_mode="like").to_pandas()[
    ["input_original", "input", "primary_name", "group_name"]
].head(10)

### 6. Fuzzy, and a caution

Scoring happens **in the engine**. The relational version pulled all 912
thousand aliases into Python and scored them with `rapidfuzz`, which also
meant an ImportError wherever that optional dependency was missing.

⚠️ **Scores are not comparable to the old ones.** Both scales are 0–100
and the default threshold is still 80, but Jaro-Winkler rewards a shared
prefix and does not reorder words. Check the threshold against your own
inputs rather than assuming the old one transfers.

In [ ]:
fuzzy = bf.report.run(
    REPORT, input_data=["TP53"], match_mode="fuzzy", similarity_threshold=90
).to_pandas()

fuzzy.sort_values("similarity_score", ascending=False)[
    ["input_original", "input", "similarity_score", "primary_name", "group_name"]
].head(12)

### 7. Narrowing by entity group

In [ ]:
for group in ("Genes", "Proteins"):
    out = bf.report.run(
        REPORT, input_data=["TP53"], match_mode="like", group_filter=group
    ).to_pandas()
    print(f"{group:10s} {len(out):>4} matches")

bf.report.run(
    REPORT, input_data=["TP53"], match_mode="like", group_filter="Proteins"
).to_pandas()[["input_original", "input", "primary_name", "group_name"]]

### 8. Export

CSV by default, with a `.provenance.json` beside it naming the bundle the
ids came from.

In [ ]:
for path in result.write("entity_filter.csv"):
    print(path)

### 9. The same thing on the command line

```bash
biofilter report run --report-name entity_filter \\
    --input TP53 --input BRCA1 \\
    --param match_mode=fuzzy --param similarity_threshold=90 \\
    --output lookup.csv
```

### 10. Quick QA

In [ ]:
expected = list(bf.report.available_columns(REPORT))
missing = [c for c in expected if c not in df.columns]

print("missing columns:", missing or "none")
print("not found:", int((df["observation"] == "not found").sum()))
print("ambiguous:", int((df["observation"] == "multiple matches").sum()))
print("bundle:", result.provenance["bundle_id"])
display(df.dtypes.to_frame("dtype"))